# Beginner challenge: stretching HeH⁺

A single energy describes only one geometry. Here we change the He–H distance and follow the ground-state potential energy curve of **HeH⁺**.

**Goals**
- Estimate the equilibrium bond length from the minimum of the curve.
- Follow the energy toward dissociation and estimate the binding energy.
- Check whether Hartree–Fock + UCCSD VQE tracks an exact reference over the whole scan.

We use the same minimal STO-3G model and Jordan–Wigner mapping as the companion tutorial. Because an unconstrained qubit diagonalization can select the wrong electron-number sector, the exact solver below uses Qiskit Nature's default electronic-structure filter.

## What to expect

HeH⁺ has two electrons (charge +1). At short distances the nuclei repel strongly; near equilibrium the energy is lowest; at long distances the curve approaches the neutral-He + proton dissociation limit. The bond length and energies depend on the basis set, so interpret the values as results for this small teaching model.

The VQE uses a noiseless statevector and a classical optimizer. Thus, differences from the exact curve measure ansatz/optimization performance here, not hardware noise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms import NumPyMinimumEigensolver, VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit_nature.second_q.algorithms import GroundStateEigensolver
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.units import DistanceUnit

In [ ]:
BASIS = "sto3g"
MAPPER = JordanWignerMapper()
CHEMICAL_ACCURACY_HA = 1.6e-3

# Use finer spacing near the minimum and a wider spacing in the dissociation tail.
distances_angstrom = np.round(
    np.r_[np.arange(0.40, 1.51, 0.05), [1.6, 1.8, 2.0, 2.5, 3.0, 4.0, 5.0, 8.0]],
    2,
)
estimator = StatevectorEstimator()

def make_problem(distance_angstrom):
    """Build the two-electron HeH⁺ electronic-structure problem at one distance."""
    return PySCFDriver(
        atom=f"He 0 0 0; H 0 0 {distance_angstrom}",
        basis=BASIS,
        charge=1,
        spin=0,
        unit=DistanceUnit.ANGSTROM,
    ).run()

def exact_total_energy(problem):
    """Return the lowest energy in the requested electron/spin sector."""
    solver = NumPyMinimumEigensolver(
        filter_criterion=problem.get_default_filter_criterion()
    )
    result = GroundStateEigensolver(MAPPER, solver).solve(problem)
    return float(result.total_energies[0])

## 1. Scan the bond distance

At every geometry, calculate the filtered exact energy and a VQE energy with the tutorial's Hartree–Fock + UCCSD ansatz. Starting each optimization at zero parameters makes the runs reproducible.

In [ ]:
rows = []
for distance in distances_angstrom:
    problem = make_problem(float(distance))
    exact_energy = exact_total_energy(problem)

    hf_state = HartreeFock(
        problem.num_spatial_orbitals, problem.num_particles, MAPPER
    )
    ansatz = UCCSD(
        problem.num_spatial_orbitals,
        problem.num_particles,
        MAPPER,
        initial_state=hf_state,
    )
    vqe = VQE(estimator, ansatz, SLSQP(maxiter=200))
    vqe.initial_point = np.zeros(ansatz.num_parameters)
    vqe_result = GroundStateEigensolver(MAPPER, vqe).solve(problem)
    vqe_energy = float(vqe_result.total_energies[0])

    rows.append(
        {
            "distance_angstrom": float(distance),
            "exact_energy_ha": exact_energy,
            "uccsd_vqe_energy_ha": vqe_energy,
            "absolute_error_ha": abs(vqe_energy - exact_energy),
        }
    )

curve = pd.DataFrame(rows)
curve.head()

In [ ]:
equilibrium_row = curve.loc[curve["exact_energy_ha"].idxmin()]

plt.figure(figsize=(8, 5))
plt.plot(curve["distance_angstrom"], curve["exact_energy_ha"], "o-", label="Exact (filtered)")
plt.plot(curve["distance_angstrom"], curve["uccsd_vqe_energy_ha"], "s--", label="UCCSD VQE")
plt.axvline(equilibrium_row["distance_angstrom"], color="gray", ls=":", label="Estimated equilibrium")
plt.xlabel("He–H distance (Å)")
plt.ylabel("Total energy (Hartree)")
plt.title("HeH⁺ potential energy curve (STO-3G)")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(curve["distance_angstrom"], 1000 * curve["absolute_error_ha"], "o-")
plt.axhline(1000 * CHEMICAL_ACCURACY_HA, color="red", ls="--", label="Chemical accuracy (1.6 mHa)")
plt.xlabel("He–H distance (Å)")
plt.ylabel("|VQE − exact| (mHa)")
plt.title("Does UCCSD VQE stay accurate along the curve?")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

max_error = curve["absolute_error_ha"].max()
worst_row = curve.loc[curve["absolute_error_ha"].idxmax()]
print(f"Largest absolute VQE error: {max_error:.3e} Ha")
print(f"Worst point: {worst_row['distance_angstrom']:.2f} Å")
print(f"Within chemical accuracy at every sampled point: {max_error < CHEMICAL_ACCURACY_HA}")

In [ ]:
dissociation_energy = float(curve.iloc[-1]["exact_energy_ha"] - equilibrium_row["exact_energy_ha"])
print(f"Estimated equilibrium distance: {equilibrium_row['distance_angstrom']:.2f} Å")
print(f"Minimum energy: {equilibrium_row['exact_energy_ha']:.8f} Ha")
print(f"Large-distance energy at {curve.iloc[-1]['distance_angstrom']:.1f} Å: {curve.iloc[-1]['exact_energy_ha']:.8f} Ha")
print(f"Estimated dissociation energy: {dissociation_energy:.6f} Ha")

## 2. Interpret the curve

For this STO-3G scan, the minimum is expected near **0.9 Å**. Beyond it, the energy rises toward the neutral-He + H⁺ limit (about **−2.8078 Ha** in this basis); the difference from the minimum is the estimated dissociation energy. At very short distances, the energy rises sharply because of nuclear repulsion.

Use the error plot and printed maximum to decide whether UCCSD VQE stays within chemical accuracy at every sampled distance. In this small two-electron model, the ansatz is expected to track the filtered exact result closely. If the curves appear dramatically different, first verify that the exact solver is filtering to the intended electron-number sector; a minimum over all qubit states is not a valid reference for HeH⁺.

### Try next
- Add more points around the minimum to refine the equilibrium estimate.
- Extend the large-distance range and check how quickly the asymptote is reached.
- Change the basis set and compare how the equilibrium and dissociation energy shift.
- Try a different optimizer or parameter initialization and see whether the VQE error changes.